In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import InputLayer, Dense
from tensorflow.keras.optimizers import Adam


In [ ]:
dataset = pd.read_csv("life_expectancy.csv")
print(dataset.head())
print(dataset.describe())

# Drop Country column (not useful for generalization)
dataset = dataset.drop(["Country"], axis=1)

# Labels = final column
labels = dataset.iloc[:, -1]
print("Labels shape:", labels.shape)


In [ ]:
# Features = all columns except the last
features = dataset.iloc[:, :-1]

# One-hot encode categorical features
features = pd.get_dummies(features)
print("Features shape after encoding:", features.shape)
print(features.head())


In [ ]:
# Train-test split
features_train, features_test, labels_train, labels_test = train_test_split(
    features, labels, test_size=0.2, random_state=23
)

# Select numeric columns automatically
numeric_cols = features.select_dtypes(include=["float64", "int64"]).columns


In [ ]:
# ColumnTransformer for scaling
ct = ColumnTransformer(
    [("standardize", StandardScaler(), numeric_cols)],
    remainder="passthrough"
)

# Fit + transform training data
features_train_scaled = ct.fit_transform(features_train)

# Transform test data
features_test_scaled = ct.transform(features_test)


In [ ]:
# Build model
my_model = Sequential(name="life_expectancy_model")

# Input layer
input_layer = InputLayer(input_shape=(features_train_scaled.shape[1],))
my_model.add(input_layer)

# Hidden layer
my_model.add(Dense(64, activation="relu"))

# Output layer
my_model.add(Dense(1))


In [ ]:
# Summary
print(my_model.summary())

# Optimizer
opt = Adam(learning_rate=0.01)

# Compile
my_model.compile(optimizer=opt, loss="mse", metrics=["mae"])

# Train
my_model.fit(features_train_scaled, labels_train, epochs=40, batch_size=1, verbose=1)

# Evaluate
res_mse, res_mae = my_model.evaluate(features_test_scaled, labels_test, verbose=0)

# Print results
print("Final MSE:", res_mse)
print("Final MAE:", res_mae)
print("Final RMSE:", np.sqrt(res_mse))
